# TV3: Bước 2 - BM25 Okapi Retrieval & Đánh giá Truy xuất Bằng chứng
## Module 7.2: Information Retrieval (BM25 vs TF-IDF Baseline)

---

### Mục tiêu:
Dựa vào Tuyên bố (Claim), tự động tìm kiếm Top-$K$ câu bằng chứng trong kho câu của bài báo tương ứng.
- Đánh giá Recall@1, Recall@3, Recall@5, MRR đối chứng giữa **Okapi BM25** và **TF-IDF**.
- Xuất danh sách Top-5 câu ứng viên cho mỗi Claim vào file `outputs/bm25_results.csv`.

In [1]:
import os
import re
import time
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

PROJECT_ROOT = Path("../..").resolve()
DEV_PATH = PROJECT_ROOT / "data/processed/common_cleaned/vifactcheck_dev_common_cleaned.csv"
CAND_PATH = OUTPUT_DIR / "evidence_candidates.csv"
OUTPUT_DIR = PROJECT_ROOT / "data/processed/retrieval"

df_dev = pd.read_csv(DEV_PATH)
df_candidates = pd.read_csv(CAND_PATH)
print(f"✓ Nạp thành công {len(df_dev):,} claims và {len(df_candidates):,} candidates.")

✓ Nạp thành công 723 claims và 12,823 candidates.


In [2]:
def tokenize_for_ir(text: str) -> list[str]:
    if not isinstance(text, str):
        return []
    return re.findall(r"\b\w+\b", text.lower())

class BM25Retriever:
    def __init__(self, candidates: list[str]):
        self.candidates = candidates
        self.tokenized_corpus = [tokenize_for_ir(doc) for doc in candidates]
        self.bm25 = BM25Okapi(self.tokenized_corpus)
        
    def retrieve(self, query: str, top_k: int = 5) -> list[tuple[str, float, int]]:
        tokenized_query = tokenize_for_ir(query)
        if not tokenized_query:
            return []
        scores = self.bm25.get_scores(tokenized_query)
        top_indices = np.argsort(scores)[::-1][:top_k]
        return [(self.candidates[idx], float(scores[idx]), int(idx)) for idx in top_indices]

def normalize_evidence_text(t: str) -> str:
    return re.sub(r"\s+", " ", str(t).lower()).strip()

def is_evidence_match(candidate: str, gold_evidence: str, min_overlap: float = 0.6) -> bool:
    if not isinstance(gold_evidence, str) or not gold_evidence.strip():
        return False
    cand_norm = normalize_evidence_text(candidate)
    gold_norm = normalize_evidence_text(gold_evidence)
    if cand_norm in gold_norm or gold_norm in cand_norm:
        return True
    c_words = set(cand_norm.split())
    g_words = set(gold_norm.split())
    if not c_words or not g_words:
        return False
    return len(c_words.intersection(g_words)) / min(len(c_words), len(g_words)) >= min_overlap

print("✓ Bộ máy BM25 và hàm kiểm thử trùng khớp Bằng chứng Vàng đã sẵn sàng.")

✓ Bộ máy BM25 và hàm kiểm thử trùng khớp Bằng chứng Vàng đã sẵn sàng.


In [3]:
TOP_K = 5
retrieved_rows = []

for _, row in df_dev.iterrows():
    claim_id = f"dev_{row['index']}"
    claim_text = str(row["Statement"])
    gold_text = str(row["Evidence"]) if pd.notna(row["Evidence"]) else ""
    
    article_candidates = df_candidates[df_candidates["claim_id"] == claim_id]["sentence_text"].tolist()
    if not article_candidates:
        continue
        
    retriever = BM25Retriever(article_candidates)
    top_results = retriever.retrieve(claim_text, top_k=TOP_K)
    
    for rank, (cand_sent, score, _) in enumerate(top_results, 1):
        is_hit = is_evidence_match(cand_sent, gold_text) if row["labels"] != 2 else False
        retrieved_rows.append({
            "claim_id": claim_id,
            "claim_index": row["index"],
            "claim": claim_text,
            "retrieved_evidence": cand_sent,
            "bm25_score": round(score, 4),
            "rank": rank,
            "is_gold": is_hit,
            "label": row["labels"],
            "gold_evidence": gold_text if row["labels"] != 2 else ""
        })

df_retrieved = pd.DataFrame(retrieved_rows)
output_bm25_path = OUTPUT_DIR / "bm25_results.csv"
df_retrieved.to_csv(output_bm25_path, index=False)

# Đánh giá hiệu năng BM25 trên tập Non-NEI
eval_df = df_retrieved[df_retrieved["label"] != 2]
n_claims = eval_df["claim_id"].nunique()

r1 = eval_df[(eval_df["rank"] == 1) & (eval_df["is_gold"] == True)]["claim_id"].nunique() / n_claims
r3 = eval_df[(eval_df["rank"] <= 3) & (eval_df["is_gold"] == True)]["claim_id"].nunique() / n_claims
r5 = eval_df[(eval_df["rank"] <= 5) & (eval_df["is_gold"] == True)]["claim_id"].nunique() / n_claims

gold_rows = eval_df[eval_df["is_gold"] == True]
mrr = (1.0 / gold_rows.groupby("claim_id")["rank"].min()).sum() / n_claims

print(f"✓ Đã lưu bảng kết quả BM25 ({len(df_retrieved):,} dòng) tại: {output_bm25_path.name}")
print("\n" + "=" * 60)
print(f"KẾT QUẢ TRUY XUẤT BM25 (Okapi):")
print(f"• Recall@1: {r1*100:.2f}%")
print(f"• Recall@3: {r3*100:.2f}%")
print(f"• Recall@5: {r5*100:.2f}%")
print(f"• MRR:      {mrr:.4f}")
print("=" * 60)
display(df_retrieved.head(6))

✓ Đã lưu bảng kết quả BM25 (3,608 dòng) tại: bm25_results.csv

KẾT QUẢ TRUY XUẤT BM25 (Okapi):
• Recall@1: 89.80%
• Recall@3: 96.40%
• Recall@5: 97.00%
• MRR:      0.9297


,claim_id,claim_index,claim,retrieved_evidence,bm25_score,rank,is_gold,label,gold_evidence
0,dev_6040,6040,"Vào tháng 4.1930 TL Nhà vua Na Uy Harald V, Vu...",Vua hề Charlie Chaplin (vua hề Sác lô) và vợ t...,56.3285,1,True,1,Vua hề Charlie Chaplin (vua hề Sác lô) và vợ t...
1,dev_6040,6040,"Vào tháng 4.1930 TL Nhà vua Na Uy Harald V, Vu...","Trong số đó, có vua hề Charlie Chaplin (vua hề...",51.3780,2,True,1,Vua hề Charlie Chaplin (vua hề Sác lô) và vợ t...
2,dev_6040,6040,"Vào tháng 4.1930 TL Nhà vua Na Uy Harald V, Vu...",Swore Oath ở khách sạn Saigon Morin năm 2004 T...,7.4638,3,False,1,Vua hề Charlie Chaplin (vua hề Sác lô) và vợ t...
3,dev_6040,6040,"Vào tháng 4.1930 TL Nhà vua Na Uy Harald V, Vu...","Phu nhân cựu Tổng thống Pháp, bà Bernadette Ch...",6.8516,4,False,1,Vua hề Charlie Chaplin (vua hề Sác lô) và vợ t...
4,dev_6040,6040,"Vào tháng 4.1930 TL Nhà vua Na Uy Harald V, Vu...","Saigon Morin, khách sạn 4 sao hàng đầu tại Huế...",5.2987,5,False,1,Vua hề Charlie Chaplin (vua hề Sác lô) và vợ t...
5,dev_5599,5599,Nhiều chi bộ chỉ mua báo đảng mà không quan tâ...,"Nhiều chi bộ mới chỉ dừng ở việc mua báo đảng,...",44.5533,1,True,1,"Việc mua, đọc, sử dụng báo, tạp chí của Đảng t..."
